# Silver Layer – Inventory Dataset

## Description
This notebook cleans and standardizes the inventory dataset from the Bronze layer.

## Source
capstone.bronze.inventory

## Target
capstone.silver.inventory

## Transformations
- Clean pharmacy_id and medication_id
- Parse price and quantity fields
- Normalize stock status
- Parse last_updated into timestamp
- Convert backorder flag into boolean
- Derive effective_on_hand_qty and is_available
- Deduplicate inventory rows by pharmacy and medication

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("capstone.bronze.inventory")

silver = (
    df
    # Clean join keys
    .withColumn("pharmacy_id", upper(trim(col("pharmacy_id"))))
    .withColumn("medication_id", upper(trim(col("medication_id"))))

    # Parse price
    .withColumn(
        "price_match",
        regexp_extract(lower(trim(col("price_raw"))), r"([0-9]+(?:\.[0-9]+)?)", 1)
    )
    .withColumn(
        "price",
        when(col("price_match") != "", col("price_match").cast("double"))
    )

    # Parse quantity
    .withColumn(
        "quantity_match",
        regexp_extract(lower(trim(col("quantity_raw"))), r"([0-9]+)", 1)
    )
    .withColumn(
        "quantity",
        when(col("quantity_match") != "", col("quantity_match").cast("int"))
    )

    # Normalize stock status with lighter logic
    .withColumn("stock_status_clean", lower(trim(col("stock_status_raw"))))
    .withColumn(
        "stock_status",
        when(col("stock_status_clean").rlike("in stock|instock|available"), "in_stock")
        .when(col("stock_status_clean").rlike("low|limited"), "low_stock")
        .when(col("stock_status_clean").rlike("out|oos|unavailable"), "out_of_stock")
        .when(col("stock_status_clean").rlike("backorder"), "backorder")
    )

    # Parse last_updated
    .withColumn(
        "last_updated_ts",
        coalesce(
            expr("try_to_timestamp(last_updated, 'yyyy-MM-dd HH:mm:ss')"),
            expr("try_to_timestamp(last_updated, 'MM/dd/yyyy HH:mm')"),
            expr("""try_to_timestamp(last_updated, "yyyy-MM-dd'T'HH:mm:ss")"""),
            expr("try_to_timestamp(last_updated, 'dd-MMM-yyyy HH:mm')"),
            expr("try_to_timestamp(last_updated, 'yyyy/MM/dd HH:mm:ss')"),
            expr("try_to_timestamp(last_updated, 'MM-dd-yyyy hh:mm a')"),
            expr("""try_to_timestamp(last_updated, "yyyy-MM-dd'T'HH:mm:ss.SSSSSS'Z'")""")
        )
    )

    # Backorder flag
    .withColumn("backorder_base", lower(trim(col("backorder_flag_raw"))))
    .withColumn(
        "is_backorder",
        when(col("backorder_base").isin("y", "yes", "true", "1", "backordered"), True)
        .when(col("backorder_base").isin("n", "no", "false", "0"), False)
    )

    # Last restock date
    .withColumn(
        "last_restock_date",
        coalesce(
            expr("try_to_date(last_restock_date_raw, 'yyyy-MM-dd')"),
            expr("try_to_date(last_restock_date_raw, 'MM/dd/yyyy')"),
            expr("try_to_date(last_restock_date_raw, 'dd-MMM-yyyy')"),
            expr("try_to_date(last_restock_date_raw, 'yyyy/MM/dd')")
        )
    )

    # Clean text fields
    .withColumn("supplier_name", initcap(trim(col("supplier_name_raw"))))
    .withColumn("availability_channel", lower(trim(col("availability_channel_raw"))))
    .withColumn("record_status", lower(trim(col("record_status_raw"))))

    # Derived fields
    .withColumn("effective_on_hand_qty", coalesce(col("quantity"), lit(0)))
    .withColumn("is_available", col("effective_on_hand_qty") > 0)
)

# Deduplicate: latest row per pharmacy + medication
window_spec = Window.partitionBy("pharmacy_id", "medication_id") \
    .orderBy(col("last_updated_ts").desc_nulls_last())

silver_dedup = (
    silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
)

final_df = silver_dedup.select(
    "pharmacy_id",
    "medication_id",
    "price",
    "quantity",
    "stock_status",
    "last_updated_ts",
    "is_backorder",
    "last_restock_date",
    "effective_on_hand_qty",
    "is_available",
    "supplier_name",
    "availability_channel",
    "record_status",
    "source_system"
)

spark.sql("DROP TABLE IF EXISTS capstone.silver.inventory")

(
    final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("capstone.silver.inventory")
)

print("Loaded: capstone.silver.inventory")
print(final_df.dtypes)